In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

CODE_REVISION = 'c8ba8eea829be0339e8d7757f8ca52dac06e1e32'
MODEL_REVISION = '7476824bc222e4ad509d286e8cae8b8d3f371129'
REPOSITORY = Path('/content/Audio8-ASR-Infinite')
ENVIRONMENT = Path('/content/audio8-venv')
PYTHON = ENVIRONMENT / 'bin/python'
MODEL_FOLDER = Path('/content/audio8-model')
INDEXED_FOLDER = Path('/content/audio8-model-indexed')
SERVER_ENV = Path('/content/audio8-server-venv')
SERVER_PYTHON = SERVER_ENV / 'bin/python'
PORT = 18190
LOG_PATH = Path('/content/audio8-server.log')

def run_command(arguments, *, env=None, cwd=None):
    result = subprocess.run(
        [str(value) for value in arguments],
        capture_output=True, text=True, env=env, cwd=cwd,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result.stdout.strip()

def run_audio8(code, *arguments):
    return run_command([PYTHON, '-u', '-c', code, *arguments])

def check_device(executable):
    result = subprocess.run([str(executable), '-c', '''
import json
import torch
device = 'cpu'
reason = 'cuda is unavailable in this python environment'
if torch.cuda.is_available():
    try:
        x = torch.ones((16, 16), device='cuda', dtype=torch.float16)
        y = x @ x
        torch.cuda.synchronize()
        device, reason = 'cuda', torch.cuda.get_device_name(0)
    except RuntimeError as error:
        reason = str(error)
print(json.dumps({'device': device, 'reason': reason, 'torch': torch.__version__}))
'''], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr or 'could not import torch in the selected environment')
    return json.loads(result.stdout.strip().splitlines()[-1])

@register_cell_magic
def audio8_gpu_only(line, cell):
    # skip the cuda server on cpu instead of pretending it supports cpu streaming.
    executable = SERVER_PYTHON if SERVER_PYTHON.exists() else PYTHON
    device = check_device(executable)
    if device['device'] != 'cuda':
        globals()['dtype_patch_ready'] = False
        print('skipped gpu-only server cell:', device['reason'])
        print('cpu model loading is available in cell 4; live cpu serving is not configured')
        return
    result = get_ipython().run_cell(cell)
    result.raise_error()

def prepare_environment(folder):
    executable = folder / 'bin/python'
    if executable.exists():
        print('using existing environment:', folder)
    elif folder.exists():
        raise RuntimeError(f'{folder} exists but python is missing; inspect before replacing')
    else:
        run_command([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', folder])
    run_command([
        executable, '-c',
        'import sys; print(sys.version); assert sys.version_info[:2] == (3, 12)',
    ])

run_command([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if not PYTHON.exists() and not ENVIRONMENT.exists():
    run_command([sys.executable, '-m', 'uv', 'python', 'install', '3.12'])
prepare_environment(ENVIRONMENT)
print('paths and environment ready')


In [ ]:
if not REPOSITORY.exists():
    run_command([
        'git', 'clone', 'https://github.com/Edge0-AI/Audio8-ASR-Infinite.git',
        REPOSITORY,
    ])
changes = run_command(['git', '-C', REPOSITORY, 'status', '--porcelain'])
if changes:
    raise RuntimeError('source has local changes; skip this cell if already installed')
run_command(['git', '-C', REPOSITORY, 'fetch', 'origin', CODE_REVISION])
run_command(['git', '-C', REPOSITORY, 'checkout', '--detach', CODE_REVISION])
run_command([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
    '-e', f'{REPOSITORY}[client]',
])
run_command([sys.executable, '-m', 'uv', 'pip', 'check', '--python', PYTHON])
run_audio8('''
import torch
import transformers
import audio8_asr_infinite
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
''')
device_info = check_device(PYTHON)
print('selected device:', device_info['device'])
print('device details:', device_info['reason'])
if device_info['device'] == 'cpu':
    print('continuing with cpu model testing; cuda streaming cells will be skipped')


In [ ]:
run_audio8('''
import sys
from huggingface_hub import snapshot_download
revision, destination = sys.argv[1:3]
print('model revision:', revision, flush=True)
path = snapshot_download(
    repo_id='Edge0/Audio8-ASR-Infinite', revision=revision, local_dir=destination,
)
print('model downloaded to:', path, flush=True)
''', MODEL_REVISION, MODEL_FOLDER)


In [ ]:
import gc
import struct

if globals().get('audio8_server') is not None and globals()['audio8_server'].poll() is None:
    raise RuntimeError('stop the streaming server before this separate model test')
globals().pop('model', None)
gc.collect()
if 'torch' in sys.modules and sys.modules['torch'].cuda.is_available():
    try:
        sys.modules['torch'].cuda.empty_cache()
    except RuntimeError:
        pass

actual_revision = run_command(['git', '-C', REPOSITORY, 'rev-parse', 'HEAD'])
if actual_revision != CODE_REVISION:
    raise RuntimeError('unexpected source revision; inspect before continuing')
parts = [
    ('model.safetensors', 'model-00001-of-00002.safetensors'),
    ('semantic_vad_heads.safetensors', 'model-00002-of-00002.safetensors'),
]
weight_map = {}
total_size = 0
for original_name, shard_name in parts:
    with (MODEL_FOLDER / original_name).open('rb') as file:
        header_size = struct.unpack('<Q', file.read(8))[0]
        if header_size > 100_000_000:
            raise RuntimeError('unexpected weight header size')
        header = json.loads(file.read(header_size))
    for name, info in header.items():
        if name == '__metadata__':
            continue
        if name in weight_map:
            raise RuntimeError(f'duplicate weight: {name}')
        weight_map[name] = shard_name
        start, end = info['data_offsets']
        total_size += end - start
expected = {
    f'semantic_vad_heads.{i}.{kind}'
    for i in range(4) for kind in ('weight', 'bias')
}
if not expected.issubset(weight_map):
    raise RuntimeError('the trained speech-pause weights are incomplete')
INDEXED_FOLDER.mkdir(exist_ok=True)
if (INDEXED_FOLDER / 'model.safetensors').exists():
    raise RuntimeError('main weight filename in prepared folder could bypass the index')

def link_existing(source, destination):
    if destination.exists():
        if not os.path.samefile(source, destination):
            raise RuntimeError(f'prepared file differs: {destination.name}')
    else:
        os.link(source, destination)

for original_name, shard_name in parts:
    link_existing(MODEL_FOLDER / original_name, INDEXED_FOLDER / shard_name)
for path in MODEL_FOLDER.iterdir():
    if (
        path.is_file() and not path.name.startswith('.')
        and not path.name.endswith('.safetensors')
        and not path.name.endswith('.safetensors.index.json')
    ):
        link_existing(path, INDEXED_FOLDER / path.name)
index = {'metadata': {'total_size': total_size}, 'weight_map': weight_map}
index_path = INDEXED_FOLDER / 'model.safetensors.index.json'
if index_path.exists():
    if json.loads(index_path.read_text()) != index:
        raise RuntimeError('prepared index differs; inspect before replacing')
else:
    index_path.write_text(json.dumps(index, indent=2), encoding='utf-8')
print('prepared folder:', INDEXED_FOLDER)
print('indexed weights:', len(weight_map))
print('indexed speech-pause weights:', len(expected))
device_info = check_device(PYTHON)
print('loading test device:', device_info['device'], '-', device_info['reason'])
run_audio8('''
import sys
from pathlib import Path
import torch
from audio8_asr_infinite.modeling.modeling_audio8_asr_infinite import (
    Audio8ASRInfiniteForConditionalGeneration,
)
device = sys.argv[2]
dtype = torch.float16 if device == 'cuda' else torch.float32
if device == 'cpu':
    # float32 uses about twice the memory of these downloaded 16-bit weights.
    available = next(
        int(line.split()[1]) * 1024
        for line in Path('/proc/meminfo').read_text().splitlines()
        if line.startswith('MemAvailable:')
    )
    estimate = int(sys.argv[3]) * 2 + 4 * 1024**3
    print('available system memory:', round(available / 1024**3, 2), 'GiB', flush=True)
    print('conservative loading estimate:', round(estimate / 1024**3, 2), 'GiB', flush=True)
    if available < estimate:
        print('CPU LOAD SKIPPED: insufficient estimated system memory; model was not loaded.')
        sys.exit(0)
    print('loading on cpu in float32; this may be slow and is not a live-speed test', flush=True)
model = Audio8ASRInfiniteForConditionalGeneration.from_pretrained(
    sys.argv[1], torch_dtype=dtype,
).to(device).eval()
if device == 'cuda':
    torch.cuda.synchronize()
print('model loaded successfully')
print('device:', next(model.parameters()).device)
print('dtype:', next(model.parameters()).dtype)
if device == 'cuda':
    print('gpu memory allocated:', round(torch.cuda.memory_allocated() / 1024**3, 2), 'GiB')
else:
    print('cpu loading test complete; live streaming is not started')
''', INDEXED_FOLDER, device_info['device'], str(total_size))


In [ ]:
%%audio8_gpu_only
if not (INDEXED_FOLDER / 'model.safetensors.index.json').exists():
    raise RuntimeError('prepare the indexed model before installing the server')
prepare_environment(SERVER_ENV)
print('installing streaming server packages; this may take several minutes', flush=True)
run_command([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', SERVER_PYTHON,
    '--torch-backend', 'cu129', 'vllm==0.27.1', 'transformers==5.13.0',
    'pybase64', '-e', f'{REPOSITORY}[client]',
])
run_command([sys.executable, '-m', 'uv', 'pip', 'check', '--python', SERVER_PYTHON])
run_command([SERVER_PYTHON, '-c', '''
import torch
from importlib.metadata import version
print('vllm:', version('vllm'))
print('transformers:', version('transformers'))
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('server gpu unavailable; subsequent server cells will skip')
'''])


In [ ]:
%%audio8_gpu_only
import ast
import difflib
import hashlib
import socket

dtype_patch_ready = False
if globals().get('audio8_server') is not None and globals()['audio8_server'].poll() is None:
    raise RuntimeError('stop the server using cell 9 before repairing source')
with socket.socket() as connection:
    if connection.connect_ex(('127.0.0.1', PORT)) == 0:
        raise RuntimeError('the server port is occupied; inspect it before patching')

relative_source = (
    'audio8_asr_infinite/vllm_serve/vllm0271/audio8_streaming_asr_realtime.py'
)
source_file = REPOSITORY / relative_source
actual_revision = subprocess.check_output(
    ['git', '-C', str(REPOSITORY), 'rev-parse', 'HEAD'], text=True,
).strip()
if actual_revision != CODE_REVISION:
    raise RuntimeError('unexpected source revision; do not apply this patch blindly')
baseline = subprocess.check_output(
    ['git', '-C', str(REPOSITORY), 'show', f'{CODE_REVISION}:{relative_source}'],
).decode('utf-8')
replacements = {
    'self._hf_language_model.to(dtype=torch.bfloat16)':
        'self._hf_language_model.to(dtype=self.vllm_config.model_config.dtype)',
    'self._hf_language_model.to(device=self._audio_device(), dtype=torch.bfloat16)':
        'self._hf_language_model.to(device=self._audio_device(), '
        'dtype=self.vllm_config.model_config.dtype)',
}
updated = baseline
for old, new in replacements.items():
    if updated.count(old) != 1:
        raise RuntimeError('expected conversion not found exactly once; no changes written')
    updated = updated.replace(old, new, 1)
ast.parse(updated)
current = source_file.read_text(encoding='utf-8')
if current not in (baseline, updated):
    raise RuntimeError('adapter has other edits; inspect them before applying this repair')

PATCH_FOLDER = Path('/content/audio8-local-patches')
PATCH_FOLDER.mkdir(exist_ok=True)
backup = PATCH_FOLDER / 'audio8_streaming_asr_realtime.py.original'
if backup.exists() and backup.read_text(encoding='utf-8') != baseline:
    raise RuntimeError('existing backup differs; preserve it and inspect before continuing')
if not backup.exists():
    backup.write_text(baseline, encoding='utf-8')
patch_text = ''.join(difflib.unified_diff(
    baseline.splitlines(keepends=True), updated.splitlines(keepends=True),
    fromfile=f'a/{relative_source}', tofile=f'b/{relative_source}',
))
(PATCH_FOLDER / 'respect-server-dtype.patch').write_text(patch_text, encoding='utf-8')
if current != updated:
    source_file.write_text(updated, encoding='utf-8')
    print('applied local dtype repair')
else:
    print('dtype repair already applied')
print(patch_text)
patched_source_hash = hashlib.sha256(updated.encode('utf-8')).hexdigest()
print('patched source sha256:', patched_source_hash)

cuda13_lib = SERVER_ENV / 'lib/python3.12/site-packages/nvidia/cu13/lib'
if not (cuda13_lib / 'libcudart.so.13').exists():
    raise RuntimeError('expected installed cuda 13 library is missing; inspect packages')
test_environment = os.environ.copy()
test_environment['LD_LIBRARY_PATH'] = os.pathsep.join(
    value for value in [str(cuda13_lib), test_environment.get('LD_LIBRARY_PATH', '')]
    if value
)
run_command([SERVER_PYTHON, '-c', '''
import torch
import vllm
import vllm._C_stable_libtorch
print('vllm:', vllm.__version__)
print('torch:', torch.__version__)
print('vllm compiled library imported successfully')
x = torch.randn(128, 128, device='cuda', dtype=torch.float16)
y = x @ x
torch.cuda.synchronize()
print('basic fp16 gpu operation passed')
'''], env=test_environment)
dtype_patch_ready = True
print('repair and import check complete; run cell 7')


In [ ]:
%%audio8_gpu_only
import hashlib
import socket
import time
import urllib.request

if not globals().get('dtype_patch_ready', False):
    raise RuntimeError('run cell 6 successfully before starting the server')
if hashlib.sha256(source_file.read_bytes()).hexdigest() != patched_source_hash:
    raise RuntimeError('source changed since the repair check; inspect and rerun cell 6')
if not SERVER_PYTHON.exists() or not (INDEXED_FOLDER / 'model.safetensors.index.json').exists():
    raise RuntimeError('server environment or indexed model is missing')
if globals().get('audio8_server') is not None and globals()['audio8_server'].poll() is None:
    raise RuntimeError('server already running; use cell 8 to inspect or cell 9 to stop')
with socket.socket() as connection:
    if connection.connect_ex(('127.0.0.1', PORT)) == 0:
        raise RuntimeError('port is occupied; inspect the existing server before continuing')

environment = os.environ.copy()
environment['LD_LIBRARY_PATH'] = test_environment['LD_LIBRARY_PATH']
environment['PYTHONPATH'] = os.pathsep.join([
    str(REPOSITORY), str(REPOSITORY / 'audio8_asr_infinite/vllm_serve'),
])
environment['VLLM_PLUGINS'] = 'audio8_streaming_asr_realtime,audio8_realtime_endpoint'
environment['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
environment['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
environment['AUDIO8_REALTIME_ROLLING_CONTEXT_TOKENS'] = '360'
environment['AUDIO8_REALTIME_DUMP_AUDIO'] = ''
environment['AUDIO8_REALTIME_DELTA_LOG_EVERY'] = '0'
environment['AUDIO8_REALTIME_UPDATE_LOG_EVERY'] = '0'
environment['AUDIO8_REALTIME_AUDIO_LOG_EVERY'] = '0'

command = [
    SERVER_PYTHON, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', INDEXED_FOLDER, '--host', '127.0.0.1', '--port', str(PORT),
    '--served-model-name', 'audio8-asr-infinite', '--trust-remote-code',
    '--dtype', 'float16', '--enforce-eager', '--max-model-len', '512',
    '--max-num-seqs', '1', '--max-num-batched-tokens', '512',
    '--kv-cache-memory-bytes', '650000000', '--gpu-memory-utilization', '0.90',
    '--scheduler-cls',
    'audio8_asr_infinite.vllm_serve.vllm0271.audio8_rolling_scheduler.Audio8RollingScheduler',
]
LOG_PATH = Path(f'/content/audio8-server-{time.time_ns()}.log')
with LOG_PATH.open('w') as log_file:
    audio8_server = subprocess.Popen(
        [str(part) for part in command], cwd=REPOSITORY, env=environment,
        stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True,
    )
audio8_process_group = audio8_server.pid
print('server process:', audio8_server.pid)
print('log:', LOG_PATH)
print('waiting up to three minutes; use cell 8 if still starting...', flush=True)
ready = False
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if audio8_server.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=2) as response:
            if response.status == 200:
                ready = True
                break
    except OSError:
        pass
    time.sleep(3)

if audio8_server.poll() is not None:
    print('server exited with code:', audio8_server.returncode)
elif ready:
    print('server health check passed')
else:
    print('server still starting; inspect with cell 8, do not start another copy')
print('\n--- recent server output ---')
print('\n'.join(LOG_PATH.read_text(errors='replace').splitlines()[-100:]))


In [ ]:
import urllib.request

if 'audio8_server' not in globals():
    print('no server process is recorded in this notebook session')
else:
    print('process id:', globals()['audio8_server'].pid)
    print('exit code (none means running):', globals()['audio8_server'].poll())
try:
    with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=3) as response:
        print('health status:', response.status)
except OSError as error:
    print('health not ready:', str(error))
if LOG_PATH.exists():
    print('log:', LOG_PATH)
    print('\n'.join(LOG_PATH.read_text(errors='replace').splitlines()[-100:]))


In [ ]:
import signal
import time

group = globals().get('audio8_process_group')
if group is None:
    print('no server process group recorded; nothing stopped')
else:
    try:
        os.killpg(group, signal.SIGTERM)
    except ProcessLookupError:
        pass
    if 'audio8_server' in globals():
        try:
            globals()['audio8_server'].wait(timeout=15)
        except subprocess.TimeoutExpired:
            pass
    deadline = time.monotonic() + 5
    while time.monotonic() < deadline:
        try:
            os.killpg(group, 0)
        except ProcessLookupError:
            break
        time.sleep(0.5)
    try:
        os.killpg(group, signal.SIGKILL)
    except ProcessLookupError:
        pass
    if 'audio8_server' in globals():
        globals()['audio8_server'].wait(timeout=10)
    audio8_process_group = None
    print('stopped the recorded server process group')
